# 06 · Kendi Verin ve Stata Komutları

Bu derste:
1. Kendi veri dosyanı (Stata `.dta`, Excel, CSV) yüklemeyi,
2. Değişken etiketlerinin raporlara nasıl yansıdığını,
3. **Stata komutlarıyla** (regress, xtreg, estat, esttab...) analiz yapmayı,
4. Hazır bir **do dosyasını** çalıştırmayı ve sonucu Word raporuna dönüştürmeyi öğreneceksin.

> Örnek dosyaları defterin içinde simüle veriden üretiyoruz; kendi çalışmanda `ek.oku("dosyan.dta")` yazman yeterli.

In [1]:
import ekonometri as ek

# örnek bir Stata dosyası hazırlayalım (kendi verin varsa bu adımı atla)
veri = ek.veri.yukle("firma")
ek.etiketle(veri, {"satis": "Yıllık satış (TL)", "calisan": "Çalışan sayısı", "sermaye": "Sermaye stoku (TL)", "arge": "Ar-Ge yoğunluğu (%)"})
ek.kaydet(veri, "firma_ornek.dta")

PosixPath('firma_ornek.dta')

## 1. Dosyayı oku

`ek.oku()` uzantıya bakar. `.dta` dosyalarında **değişken etiketleri** ve **değer etiketleri** korunur; Excel/CSV'de Türkçe biçimler (ondalık virgül, `;` ayırıcı, Türkçe karakterler) otomatik tanınır.

In [2]:
veri = ek.oku("firma_ornek.dta")
veri.attrs["etiketler"]

{'satis': 'Yıllık satış (TL)',
 'calisan': 'Çalışan sayısı',
 'sermaye': 'Sermaye stoku (TL)',
 'arge': 'Ar-Ge yoğunluğu (%)'}

## 2. Stata komutlarıyla analiz

In [3]:
s = ek.Stata(veri)
s("describe")
s("summarize satis calisan sermaye arge");

Gözlem: 1500  Değişken: 8
Değişken            Tür           Etiket
────────────────────────────────────────────────────────────
firma               metin         
yil                 sayısal       
satis               sayısal       Yıllık satış (TL)
calisan             sayısal       Çalışan sayısı
sermaye             sayısal       Sermaye stoku (TL)
arge                sayısal       Ar-Ge yoğunluğu (%)
ihracatci           kategorik     
sektor              kategorik     
            N     Ortalama   Std. sapma  En küçük     En büyük
satis    1500 1.219.904,00 1.137.442,31 90.000,00 7.751.000,00
calisan  1500      52,4713      42,2234    9,0000     347,0000
sermaye  1500   149.318,67    99.912,74 19.000,00   666.000,00
arge     1500       1,8101       1,0244    0,0000       5,3200


In [4]:
s("gen lns = ln(satis)")
s("gen lnl = ln(calisan)")
s("gen lnk = ln(sermaye)")
s("xtset firma yil")

Panel değişkeni: firma (dengeli, 150 birim)
Zaman değişkeni: yil, 2015 – 2024


Sabit ve rassal etkiler, ardından Hausman testi — Stata'daki sırayla:

In [5]:
s("quietly xtreg lns lnl lnk arge i.yil, fe")
s("estimates store fe")
s("quietly xtreg lns lnl lnk arge i.yil, re")
s("estimates store re")
s("hausman fe re");

Hausman testi (sabit etkiler – rassal etkiler): istatistik = 91,656 [12], p < 0,0001
   H0: Birim etkileri açıklayıcı değişkenlerle ilişkisizdir (rassal etkiler tutarlıdır).
   p < 0,05: H0 reddedildi — birim etkileri açıklayıcı değişkenlerle ilişkili; rassal etkiler tutarsız, sabit etkiler tercih edilir. (Uyarı: kovaryans farkı pozitif tanımlı değil; genelleştirilmiş ters kullanıldı, sonuç temkinli yorumlanmalı.)


In [6]:
s("xtreg lns lnl lnk arge i.yil, fe vce(cluster firma)");

══════════════════════════════════════════════════════════════════════════════
 Panel Veri — Sabit etkiler · bağımlı değişken: ln(satis)
══════════════════════════════════════════════════════════════════════════════
 Gözlem: 1500     Birim: 150    Dönem: 10    Panel: dengeli
 R² içsel: 0,6624    arası: 0,7714    genel: 0,7586    F(12, 149) = 228,36 (p <0,0001)
 Standart hatalar: birimlere göre kümelenmiş (150 küme; değişen varyans ve birim içi otokorelasyona dayanıklı)
──────────────────────────────────────────────────────────────────────────────
 Değişken                         Katsayı   Std. hata        t         p
──────────────────────────────────────────────────────────────────────────────
 Sabit                             7,6239      0,2653    28,74   <0,0001 ***
 yil: 2016                         0,0362      0,0142     2,55    0,0117 **
 yil: 2017                         0,0362      0,0138     2,63    0,0095 ***
 yil: 2018                         0,0582      0,0145     4,02   

`gen lns = ln(satis)` ile üretilen değişkenler hatırlandığı için katsayılar **esneklik** olarak yorumlanır:

In [7]:
s("yorum");

📊 Yorum — Yıllık satış (TL) (logaritması)

▸ Model geneli
  • Sabit etkiler modeli: 150 birim, 10 dönem, 1.500 gözlem (dengeli panel).
  • R² içsel = 0,662 (birim içi zamana bağlı değişimin açıklanan kısmı), birimler arası = 0,771, genel = 0,759.
  • Standart hatalar: birimlere göre kümelenmiş (150 küme; değişen varyans ve birim içi otokorelasyona dayanıklı).

▸ Katsayılar (diğer değişkenler sabitken)
  • 'yil' değişkeninde '2016' grubunun Yıllık satış (TL) değeri, referans grupla ('2015') karşılaştırıldığında ortalama %3,7 daha yüksek — %5 düzeyinde istatistiksel olarak anlamlı (p = 0,0117).
  • 'yil' değişkeninde '2017' grubunun Yıllık satış (TL) değeri, referans grupla ('2015') karşılaştırıldığında ortalama %3,7 daha yüksek — %1 düzeyinde istatistiksel olarak anlamlı (p = 0,0095).
  • 'yil' değişkeninde '2018' grubunun Yıllık satış (TL) değeri, referans grupla ('2015') karşılaştırıldığında ortalama %6,0 daha yüksek — %1 düzeyinde istatistiksel olarak anlamlı (p < 0,0001).
  • 'yil' 

## 3. Do dosyası

Elindeki do dosyalarını olduğu gibi çalıştırabilirsin. Yorumlar (`*`, `//`), satır devamı (`///`) ve `quietly` desteklenir. Önce örnek bir do dosyası yazalım:

In [8]:
do_metni = 'use firma_ornek.dta, clear\ngen lns = ln(satis)\ngen lnk = ln(sermaye)\n* havuzlanmış modeller\nquietly regress lns lnk, vce(cluster firma)\nestimates store m1\nquietly regress lns lnk arge i.ihracatci, ///\n    vce(cluster firma)\nestimates store m2\nesttab m1 m2\n'
with open("analiz.do", "w", encoding="utf-8") as f:
    f.write(do_metni)
print(do_metni)

use firma_ornek.dta, clear
gen lns = ln(satis)
gen lnk = ln(sermaye)
* havuzlanmış modeller
quietly regress lns lnk, vce(cluster firma)
estimates store m1
quietly regress lns lnk arge i.ihracatci, ///
    vce(cluster firma)
estimates store m2
esttab m1 m2



In [9]:
ek.Stata().do("analiz.do");

(1.500 gözlem, 8 değişken yüklendi: firma_ornek.dta)
════════════════════════════════════════════════════
Değişken                            m1            m2
────────────────────────────────────────────────────
ln(Sermaye stoku (TL))       0,8752***     0,8758***
                              (0,0678)      (0,0671)
ihracatci: Evet                               0,1549
                                            (0,1027)
Ar-Ge yoğunluğu (%)                        0,0657***
                                            (0,0161)
Sabit                        3,4131***     3,2308***
                              (0,7945)      (0,7882)
────────────────────────────────────────────────────
Gözlem sayısı                    1.500         1.500
R²                               0,436         0,456
Düzeltilmiş R²                   0,436         0,455
F                            166,51***      64,87***
AIC                            2.834,3       2.783,3
BIC                            2.844,9       2

## 4. Türkçe tez raporu

Son tahmin edilen model için Yöntem + Bulgular metni ve tablolar (etiketler kullanılır):

In [10]:
s("quietly xtreg lns lnl lnk arge i.yil, fe vce(cluster firma)")
rapor = s.son.rapor()
print(rapor.paragraflar[0])

Çalışmada 150 birim ve 10 dönemden oluşan dengeli panel veri seti (1.500 gözlem) kullanılmış, model sabit etkiler (içsel / within) tahmincisiyle tahmin edilmiştir (Baltagi, 2021). Standart hatalar: birimlere göre kümelenmiş (150 küme; değişen varyans ve birim içi otokorelasyona dayanıklı). Hesaplar ekonometri-tr 0.8.0 ile yapılmıştır.


Word dosyası için: `s("rapor bulgular.docx")`. Panel modelinin tam tez bölümü (model seçim testleriyle) için Python fonksiyonu: `ek.panel_analizi(...).rapor("panel.docx")`.

Tüm komutlar, Stata ile uyum kuralları ve bilinen farklar: [REHBER.md](../REHBER.md)
